# Cricket Fantasy Agent System

## Phase 1: Configuration & Shared Setup

**Purpose**: Initialize LLM, API credentials, and shared utilities for caching and API calls.

**Key Components**:
- LangChain LLM initialization (Groq)
- Shared HEADERS for all Cricbuzz API calls
- Cache management functions (`load_cache()`, `save_cache()`)
- Single source of truth for API configuration

In [16]:
# ==========================================
# PHASE 1: IMPORTS WITH COMPREHENSIVE ERROR HANDLING
# ==========================================

import os
import json
import logging
from pathlib import Path
from typing import Optional,Union

import pytz
IST = pytz.timezone('Asia/Kolkata')
# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Load environment variables with error handling
try:
    from dotenv import load_dotenv
    load_dotenv()
    logger.info("✅ Environment variables loaded from .env file")
except ImportError as e:
    logger.warning(f"⚠️ python-dotenv not installed: {e}. Using fallback environment.")
except Exception as e:
    logger.error(f"❌ Failed to load .env file: {e}")

# Initialize environment variables with fallback values
env_vars = {
    'GOOGLE_API_KEY': 'GOOGLE_API_KEY',
    'TAVILY_API_KEY': 'TAVILY_API_KEY',
    'GROQ_API_KEY': 'GROQ_API_KEY',
    'RAPIDAPI_KEY': 'RAPIDAPI_KEY',
    'WEATHER_API_KEY': 'WEATHER_API_KEY'
}

for var_name, var_key in env_vars.items():
    try:
        value = os.getenv(var_key)
        if value:
            os.environ[var_name] = value
            logger.info(f"✅ {var_name} loaded successfully")
        else:
            logger.warning(f"⚠️ {var_name} not found in environment. Using default.")
    except Exception as e:
        logger.error(f"❌ Error loading {var_name}: {e}")

# Initialize LangChain chat model with error handling
try:
    from langchain.chat_models import init_chat_model
    model = init_chat_model(model='meta-llama/llama-4-scout-17b-16e-instruct', model_provider="groq",temperature=1)
    logger.info("✅ LangChain model initialized successfully (Groq)")
except ImportError as e:
    logger.error(f"❌ Failed to import init_chat_model: {e}")
    raise SystemExit("Required dependency 'langchain' not found. Please install it.")
except Exception as e:
    logger.error(f"❌ Failed to initialize chat model: {e}")
    raise SystemExit(f"Chat model initialization failed: {e}")
# model = init_chat_model(model='gemini-3-pro-preview',model_provider="google_genai",base_url="https://api.ai-gateway.tigeranalytics.com")

# ==========================================
# SHARED CONFIGURATION (For All APIs & Caching)
# ==========================================
from datetime import datetime, timedelta

# API Headers - Load from .env for security
HEADERS = {
    "x-rapidapi-key": os.getenv('RAPIDAPI_KEY', ''),
    "x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
}

CACHE_FILE = "cricket_cache.json"

# ==========================================
# CACHE CLASS - Optimized Cache Management with Error Handling
# ==========================================
class CacheManager:
    """Efficient cache manager with comprehensive error handling and cleanup mechanism."""

    def __init__(self, cache_file: str):
        self.cache_file = cache_file
        self.cache_data = self._load()
        logger.info(f"✅ CacheManager initialized with {self.size()} cached entries")

    def _load(self) -> dict:
        """Load cache from file (called once on init)."""
        if not os.path.exists(self.cache_file):
            logger.info(f"ℹ️ Cache file '{self.cache_file}' not found. Starting fresh.")
            return {}

        try:
            with open(self.cache_file, "r") as f:
                data = json.load(f)
                logger.info(f"✅ Cache loaded: {len(data)} entries")
                return data
        except json.JSONDecodeError as e:
            logger.error(f"❌ Cache file corrupted (JSON decode error): {e}. Starting fresh.")
            return {}
        except IOError as e:
            logger.error(f"❌ Failed to read cache file: {e}. Starting fresh.")
            return {}
        except Exception as e:
            logger.error(f"❌ Unexpected error loading cache: {e}. Starting fresh.")
            return {}

    def _save(self) -> None:
        """Save cache to file with error handling."""
        temp_file = f"{self.cache_file}.tmp"
        try:
            # Write to temp file first (atomic write pattern)
            with open(temp_file, "w") as f:
                json.dump(self.cache_data, f, indent=4)
            # Rename temp file to actual file
            os.replace(temp_file, self.cache_file)
            logger.debug(f"✅ Cache saved successfully ({self.size()} entries)")
        except IOError as e:
            logger.error(f"❌ Failed to save cache: {e}")
            # Clean up temp file if it exists
            try:
                if os.path.exists(temp_file):
                    os.remove(temp_file)
            except:
                pass
        except Exception as e:
            logger.error(f"❌ Unexpected error saving cache: {e}")
            try:
                if os.path.exists(temp_file):
                    os.remove(temp_file)
            except:
                pass

    def get(self, key: str) -> Optional[dict]:
        """Get value from cache. Returns None if not found."""
        try:
            return self.cache_data.get(key)
        except Exception as e:
            logger.warning(f"⚠️ Error retrieving cache key '{key}': {e}")
            return None

    def set(self, key: str, value) -> None:
        """Set value in cache with timestamp. Auto-saves to file."""
        try:
            self.cache_data[key] = {
                "data": value,
                "timestamp": datetime.now().isoformat()
            }
            self._save()
        except TypeError as e:
            logger.error(f"❌ Failed to serialize cache value for key '{key}': {e}")
        except Exception as e:
            logger.error(f"❌ Unexpected error setting cache key '{key}': {e}")

    def exists(self, key: str) -> bool:
        """Check if key exists in cache."""
        try:
            return key in self.cache_data
        except Exception as e:
            logger.warning(f"⚠️ Error checking cache key '{key}': {e}")
            return False

    def cleanup(self, days: int = 7) -> int:
        """Remove cache entries older than X days. Returns count removed."""
        keys_to_remove = []
        try:
            cutoff_date = datetime.now() - timedelta(days=days)

            for key, entry in self.cache_data.items():
                try:
                    if isinstance(entry, dict) and "timestamp" in entry:
                        entry_date = datetime.fromisoformat(entry["timestamp"])
                        if entry_date < cutoff_date:
                            keys_to_remove.append(key)
                except ValueError as e:
                    logger.warning(f"⚠️ Invalid timestamp format for key '{key}': {e}. Marking for removal.")
                    keys_to_remove.append(key)
                except Exception as e:
                    logger.warning(f"⚠️ Error processing entry '{key}': {e}")

            # Remove expired entries
            for key in keys_to_remove:
                try:
                    del self.cache_data[key]
                except KeyError:
                    pass

            # Save if any entries were removed
            if keys_to_remove:
                self._save()
                logger.info(f"🧹 Cleanup: Removed {len(keys_to_remove)} expired cache entries (older than {days} days)")
            else:
                logger.info(f"ℹ️ Cleanup: No expired entries found (threshold: {days} days)")

            return len(keys_to_remove)
        except Exception as e:
            logger.error(f"❌ Unexpected error during cleanup: {e}")
            return 0

    def clear(self) -> None:
        """Clear all cache entries."""
        try:
            old_size = self.size()
            self.cache_data = {}
            self._save()
            logger.info(f"🗑️ Cache cleared ({old_size} entries removed)")
        except Exception as e:
            logger.error(f"❌ Failed to clear cache: {e}")

    def size(self) -> int:
        """Get number of cache entries."""
        try:
            return len(self.cache_data)
        except Exception as e:
            logger.warning(f"⚠️ Error getting cache size: {e}")
            return 0

# Initialize global cache instance (used by all tools)
try:
    cache = CacheManager(CACHE_FILE)
    logger.info("✅ Cache system initialized successfully")
except Exception as e:
    logger.error(f"❌ Failed to initialize cache system: {e}")
    # Create a fallback in-memory cache
    class MemoryCacheManager:
        def __init__(self):
            self.cache_data = {}
        def exists(self, key): return key in self.cache_data
        def get(self, key): return self.cache_data.get(key)
        def set(self, key, value): self.cache_data[key] = {"data": value, "timestamp": datetime.now().isoformat()}
        def cleanup(self, days=7): return 0
        def clear(self): self.cache_data = {}
        def size(self): return len(self.cache_data)

    cache = MemoryCacheManager()
    logger.warning("⚠️ Using fallback in-memory cache (file cache unavailable)")

2026-03-28 10:11:34,656 - INFO - ✅ Environment variables loaded from .env file


2026-03-28 10:11:34,657 - INFO - ✅ GOOGLE_API_KEY loaded successfully
2026-03-28 10:11:34,658 - INFO - ✅ TAVILY_API_KEY loaded successfully
2026-03-28 10:11:34,659 - INFO - ✅ GROQ_API_KEY loaded successfully
2026-03-28 10:11:34,660 - INFO - ✅ RAPIDAPI_KEY loaded successfully
2026-03-28 10:11:34,661 - INFO - ✅ WEATHER_API_KEY loaded successfully
2026-03-28 10:11:34,780 - INFO - ✅ LangChain model initialized successfully (Groq)
2026-03-28 10:11:34,789 - INFO - ✅ Cache loaded: 0 entries
2026-03-28 10:11:34,790 - INFO - ✅ CacheManager initialized with 0 cached entries
2026-03-28 10:11:34,791 - INFO - ✅ Cache system initialized successfully


In [17]:
import gspread
from oauth2client.service_account import ServiceAccountCredentials

# 1. Setup Connection
scope = ["https://spreadsheets.google.com/feeds", "https://www.googleapis.com/auth/drive"]
creds = ServiceAccountCredentials.from_json_keyfile_name("/mnt/c/workspaces/agent_project/myagentproject-491107-729a12dc5f1d.json", scope)
client = gspread.authorize(creds)

# 2. Open your Sheet (Use the name you gave your Google Sheet)
# Example: "Cricket_Project_DB"
spreadsheet_name = "Cricket_Project_DB."
spreadsheet = client.open(spreadsheet_name)
player_agent_sheet   = spreadsheet.worksheet("Player_agent")

## Phase 2: Tool Definitions (Cached)

**Purpose**: Define all LangChain tools with built-in caching to avoid redundant API calls.

**Tools Defined**:
- `get__weathr()` - Fetch weather for a specific city/date (Visual Crossing API)
- `get_match_venue()` - Fetch stadium info & pitch history (Tavily Search + caching)
- `get_match_environment()` - Fetch current pitch report (Tavily Search + caching)
- `get_cricket_squads_by_match()` - Fetch playing XI for both teams (Cricbuzz API + caching)

**Cache Strategy**: All tools cache their responses using shared `cricket_cache.json` file

In [18]:
# ==========================================
# PHASE 2: TOOL IMPORTS WITH ERROR HANDLING
# ==========================================

# Import tools with comprehensive error handling
try:
    from langchain.tools import tool
    logger.info("✅ LangChain tools imported successfully")
except ImportError as e:
    logger.error(f"❌ Failed to import LangChain tools: {e}")
    raise SystemExit("Required dependency 'langchain' not found.")

try:
    from langchain_tavily import TavilySearch
    logger.info("✅ Tavily Search imported successfully")
except ImportError as e:
    logger.error(f"❌ Failed to import Tavily Search: {e}")
    raise SystemExit("Required dependency 'langchain-tavily' not found.")

try:
    import requests
    logger.info("✅ Requests library imported successfully")
except ImportError as e:
    logger.error(f"❌ Failed to import requests: {e}")
    raise SystemExit("Required dependency 'requests' not found.")

# Initialize Tavily client with error handling
try:
    tavily_client = TavilySearch()
    logger.info("✅ Tavily Search client initialized successfully")
except Exception as e:
    logger.error(f"❌ Failed to initialize Tavily client: {e}")
    tavily_client = None

# ==========================================
# CACHED TOOL DEFINITIONS (ALL TOOLS - Used by Both Agents)
# ==========================================


def get__weathr(city: str, date: str) -> str:
    """
    Fetches the weather for a specific city on a specific date with caching.
    Input 'city' should be a string (e.g., 'Chennai').
    Input 'date' should be in YYYY-MM-DD format.
    """
    cache_key = f"weather_{city}_{date}"

    try:
        # Check cache first (no file I/O)
        if cache.exists(cache_key):
            try:
                cached_value = cache.get(cache_key)
                logger.debug(f"✅ Weather cache hit for {city} on {date}")
                return f"[CACHED] {cached_value['data']}"
            except Exception as e:
                logger.warning(f"⚠️ Failed to retrieve cached weather: {e}")

        API_KEY = os.getenv('WEATHER_API_KEY')
        if not API_KEY:
            logger.error("❌ WEATHER_API_KEY not configured in environment")
            return "Error: Weather API key not configured. Please set WEATHER_API_KEY in .env"

        url = f"https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/{city}/{date}?unitGroup=metric&key={API_KEY}&contentType=json"

        response = requests.get(url, timeout=10)

        if response.status_code == 401:
            logger.error("❌ Weather API authentication failed (401)")
            return "Error: Weather API authentication failed. Check WEATHER_API_KEY."
        elif response.status_code == 404:
            logger.warning(f"⚠️ Weather data not found for {city} on {date}")
            return f"Weather data not available for {city} on {date}"
        elif response.status_code != 200:
            logger.error(f"❌ Weather API returned status {response.status_code}")
            return f"Error: API returned status code {response.status_code}"

        data = response.json()
        day_data = data.get('days', [{}])[0]
        temp = day_data.get('temp')
        conditions = day_data.get('conditions')
        description = day_data.get('description', 'No description available.')

        result = (f"Weather in {city} on {date}: "
                  f"Temperature: {temp}°C, "
                  f"Conditions: {conditions}. "
                  f"Summary: {description}")

        # Store in cache (single write operation)
        cache.set(cache_key, result)
        logger.info(f"✅ Weather for {city} retrieved and cached")
        return result

    except requests.Timeout:
        logger.error(f"❌ Weather API request timed out for {city}/{date}")
        return f"Error: Weather API request timed out (10s timeout)"
    except requests.ConnectionError as e:
        logger.error(f"❌ Connection error fetching weather: {e}")
        return f"Error: Connection error to weather service: {str(e)}"
    except requests.RequestException as e:
        logger.error(f"❌ Weather API request failed: {e}")
        return f"Error: Failed to fetch weather data: {str(e)}"
    except (KeyError, IndexError, ValueError) as e:
        logger.error(f"❌ Failed to parse weather API response: {e}")
        return f"Error: Invalid weather API response format"
    except Exception as e:
        logger.error(f"❌ Unexpected error in get__weathr: {e}")
        return f"Error: Unexpected error fetching weather: {str(e)}"


def get_match_venue(match: str) -> str:
    """
    REQUIRED: Fetches the official stadium name, city, and critical venue history with caching.
    Includes average first innings scores and pitch history.
    """
    cache_key = f"venue_{match.lower().replace(' ', '_')}"

    try:
        if cache.exists(cache_key):
            try:
                cached_value = cache.get(cache_key)
                logger.debug(f"✅ Venue cache hit for {match}")
                return f"[CACHED] {cached_value['data']}"
            except Exception as e:
                logger.warning(f"⚠️ Failed to retrieve cached venue: {e}")

        if tavily_client is None:
            logger.error("❌ Tavily client not initialized")
            return "Error: Tavily search service not available"

        query = (f"official venue for {match} today, average first innings score men and women T20, "
                 f"is it a batting or bowling pitch, stadium history and record")

        result = str(tavily_client.invoke(query))
        cache.set(cache_key, result)
        logger.info(f"✅ Venue information retrieved and cached for {match}")
        return result

    except Exception as e:
        logger.error(f"❌ Error fetching venue for {match}: {e}")
        return f"Error: Failed to fetch venue information: {str(e)}"


def get_match_environment(location: str) -> str:
    """
    Fetches pitch report status for a specific stadium with caching.
    Use this to determine if the conditions favor bowlers or batsmen.
    """
    cache_key = f"pitch_{location.lower().replace(' ', '_')}"

    try:
        if cache.exists(cache_key):
            try:
                cached_value = cache.get(cache_key)
                logger.debug(f"✅ Pitch cache hit for {location}")
                return f"[CACHED] {cached_value['data']}"
            except Exception as e:
                logger.warning(f"⚠️ Failed to retrieve cached pitch info: {e}")

        if tavily_client is None:
            logger.error("❌ Tavily client not initialized")
            return "Error: Tavily search service not available"

        query = f"current pitch report for cricket match at {location}"
        result = str(tavily_client.invoke(query))
        cache.set(cache_key, result)
        logger.info(f"✅ Pitch report retrieved and cached for {location}")
        return result

    except Exception as e:
        logger.error(f"❌ Error fetching pitch environment for {location}: {e}")
        return f"Error: Failed to fetch pitch environment: {str(e)}"
import json


def get_cricket_squads_by_match(match_id: int) -> str:
    """
    Retrieves the playing 11 and player IDs for both teams as a structured dictionary.
    Returns a JSON string for easy parsing by other tools.
    """
    cache_key = f"squad_dict_{match_id}"

    try:
        # 1. Check Cache
        if cache.exists(cache_key):
            cached_value = cache.get(cache_key)
            return json.dumps(cached_value['data']) # Return as JSON string

        # 2. Get Match Info
        info_url = f"https://cricbuzz-cricket.p.rapidapi.com/mcenter/v1/{match_id}"
        info_res = requests.get(info_url, headers=HEADERS, timeout=10)
        match_data = info_res.json()

        # Initialize dictionary structure
        squad_dict = {
            "match_id": match_id,
            "teams": {
                "team1": {"name": match_data['team1']['teamname'], "id": match_data['team1']['teamid'], "players": []},
                "team2": {"name": match_data['team2']['teamname'], "id": match_data['team2']['teamid'], "players": []}
            }
        }

        # 3. Fetch Squads for both Teams
        for team_key in ["team1", "team2"]:
            t_id = squad_dict["teams"][team_key]["id"]
            squad_url = f"https://cricbuzz-cricket.p.rapidapi.com/mcenter/v1/{match_id}/team/{t_id}"
            squad_res = requests.get(squad_url, headers=HEADERS, timeout=10)
            s_data = squad_res.json()

            for group in s_data.get('player', []):
                if group.get('category') == "playing XI":
                    for p in group.get('player', []):
                        # Append player dict instead of string concatenation
                        squad_dict["teams"][team_key]["players"].append({
                            "id": str(p.get('id')),
                            "name": p.get('name'),
                            "role": p.get('role')
                        })

        # 4. Save to Cache and Return JSON
        cache.set(cache_key, squad_dict)
        return json.dumps(squad_dict, indent=2)

    except Exception as e:
        return json.dumps({"error": str(e)})


2026-03-28 10:11:37,371 - INFO - ✅ LangChain tools imported successfully
2026-03-28 10:11:37,373 - INFO - ✅ Tavily Search imported successfully
2026-03-28 10:11:37,374 - INFO - ✅ Requests library imported successfully
2026-03-28 10:11:37,375 - INFO - ✅ Tavily Search client initialized successfully


## Phase 3: Eco-Scout Agent (Environment-Based Selection)

**Purpose**: Create an AI agent that selects fantasy teams based on environmental factors (weather, pitch, venue).

**Agent Strategy**:
1. Fetch venue details (stadium name, city, pitch history)
2. Fetch weather conditions (temperature, conditions, description)
3. Analyze pitch behavior (batting vs bowling friendly)
4. Retrieve playing squads
5. Select 11 players based on environmental factors ONLY

**Tool Execution Order**: venue → weather → pitch → squads  
**Selection Criteria**: Weather + Pitch conditions (environment-driven, not form-driven)

In [19]:
from typing import List
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ToolStrategy
from typing import List, Optional
from pydantic import BaseModel, Field

class FantasyPlayer(BaseModel):
    """Information about a single selected fantasy player."""
    player_id: str = Field(description="The unique ID of the player from the squad tool")
    name: str = Field(description="Full name of the player")
    team: str = Field(description="Team name of the player")
    role: str = Field(description="Role: BAT, BOWL, AR, or WK")
    scout_logic: str = Field(description="Detailed reasoning for selecting this player")

# 2. CREATE THIS NEW CONTAINER CLASS
class FantasyTeam(BaseModel):
    """The full squad of 11 players."""
    reasoning_summary: str = Field(description="Overall logic for the whole team")
    players: List[FantasyPlayer] = Field(description="A list of exactly 11 selected players")





In [20]:
# ==========================================
# REASONING SNAPSHOT STRATEGY
# ==========================================
# Cache the ENTIRE tool context as one snapshot instead of individual tool outputs
# This eliminates redundant API calls and tool-choice errors

def generate_match_context_key(match: str, scout_type: str) -> str:
    """
    Generates a unique key like: 'eco_match_context_nz_vs_rsa'
    or 'form_match_context_nz_vs_rsa'
    """
    clean_match = match.lower().replace(' ', '_')
    return f"{scout_type}_match_context_{clean_match}"

def get_or_generate_match_context(match_id:int,match: str, venue: str = None,city:str =None) -> dict:
    """
    REASONING SNAPSHOT STRATEGY:
    - First call: Generate context by running all tools ONCE
    - Subsequent calls: Return cached context (NO tool calls)

    Returns: {"context": full_match_context_string, "is_cached": bool}
    """
    cache_key = generate_match_context_key(match, "eco")

    # Check if context already exists in cache
    if cache.exists(cache_key):
        try:
            cached_data = cache.get(cache_key)
            logger.info(f"✅ [SNAPSHOT] Reusing cached match context for {match}")
            return {
                "context": cached_data['data']['context'],
                "is_cached": True,
                "cached_at": cached_data.get('timestamp', 'unknown')
            }
        except Exception as e:
            logger.warning(f"⚠️ Failed to retrieve cached context: {e}")

    # No cache: Generate fresh context by calling all tools
    logger.info(f"📸 [SNAPSHOT] Generating fresh match context for {match}")

    try:
        # Call all tools in sequence to build complete context
        # venue_data = get_match_venue(match)
        squads_data = get_cricket_squads_by_match(match_id)

        # For Eco-Scout: also get weather and pitch
        weather_data = get__weathr(city, datetime.now().strftime("%Y-%m-%d"))
        combined_location = f"venue_name {venue} + city {city}" # Placeholder city
        pitch_data = get_match_environment(location= combined_location)

        # Combine all data into one comprehensive context
        full_context = f"""
=== MATCH REASONING SNAPSHOT ===
Generated at: {datetime.now(IST).strftime('%Y-%m-%d %H:%M:%S IST')}
Match: {match}


--- PLAYING SQUADS ---
{squads_data}

--- WEATHER CONDITIONS ---
{weather_data}

--- PITCH ANALYSIS ---
{pitch_data}

=== END SNAPSHOT ===
"""

        # Cache the entire context as a snapshot
        cache.set(cache_key, {"context": full_context})
        logger.info(f"✅ [SNAPSHOT] Cached complete match context ({len(full_context)} chars)")

        return {
            "context": full_context,
            "is_cached": False,
            "generated_at": datetime.now(IST).strftime('%Y-%m-%d %H:%M:%S IST')
        }

    except Exception as e:
        logger.error(f"❌ [SNAPSHOT] Failed to generate match context: {e}")
        return {
            "context": f"Error generating context: {str(e)}",
            "is_cached": False,
            "error": str(e)
        }



In [21]:
from langchain.agents import create_agent

# ==========================================
# ECO-SCOUT AGENT WITH REASONING SNAPSHOT
# ==========================================

# eco_agent_tools = [get__weathr, get_match_venue, get_match_environment, get_cricket_squads_by_match]

eco_scout_system_prompt = """Your Role:
You are the "Eco-Scout" Agent — a Cricket Fantasy Analyst whose decisions are STRICTLY based on environmental factors.

CRITICAL: You are operating in SNAPSHOT MODE. A match context snapshot has been pre-generated and cached.
You do NOT need to call tools. Instead, use the provided match context to analyze environment and select 11 players.

--- MATCH CONTEXT SNAPSHOT ---
{match_context_snapshot}
--- END SNAPSHOT ---

SELECTION RULES:
1. Use ONLY the data in the snapshot above
2. Environmental reasoning: Focus on weather, pitch behavior, and venue characteristics
3. Selection driven by: temperature, humidity, pitch type (bowling vs batting friendly)
4. Do NOT consider: player form, reputation, or past performance
5. Do NOT invent players - use only players from the squads provided in snapshot

ROSTER CONSTRAINTS (STRICT):
- Exactly 11 players
- 1 Wicket-Keeper (must have WK role from squads)
- 4 Bowlers (must have BOWL role from squads)
- 6 Batsmen/All-rounders (BAT or AR roles from squads)

OUTPUT INSTRUCTIONS:
1. Provide reasoning_summary: Explain how weather/pitch influenced your selection
2. Select 11 players from the snapshot squads
3. CRITICAL: Include 'player_id' for EVERY player (copy from snapshot)
4. Create FantasyTeam JSON with:
   - reasoning_summary: 3-4 lines explaining environmental logic
   - players: List of 11 FantasyPlayer objects (id, name, team, role, scout_logic)

FAILURE RULE:
If any player is missing 'player_id' or does not exist in snapshot squads, selection is INVALID.
"""

def create_eco_scout_with_snapshot(match_details: Union[str, dict], agent_type: str = "eco"):
    """
    Factory function to create Eco-Scout agent or return context for Merging.
    Handles both cached and fresh context generation.
    """
    logger.info(f"🔍 Eco-Scout initializing for: {match_details['match']} | Mode: {agent_type}")

    # 1. Get or generate match context snapshot (Pitch, Weather, Venue)
    context_result = get_or_generate_match_context(
        match_details['id'],
        match_details['match'],
        match_details['venue'],
        match_details['city']
    )
    match_context = context_result['context']

    # --- NEW LOGIC FOR OVERALL SCOUT ---
    # Return raw context string if we are only gathering data for the Merge Agent
    if agent_type == "overall":
        logger.info(f"📤 Returning match_context only for Overall-Scout merge.")
        return match_context

    # --- ORIGINAL LOGIC FOR ECO SCOUT ---
    is_cached = context_result.get('is_cached', False)
    status = "📦 [CACHED]" if is_cached else "📸 [FRESH]"
    logger.info(f"{status} Eco-Scout context ready ({len(match_context)} chars)")

    # Inject snapshot into system prompt
    final_system_prompt = eco_scout_system_prompt.format(
        match_context_snapshot=match_context
    )

    # Create agent WITHOUT tools (snapshot contains all data)
    agent = create_agent(
        model=model,
        tools=[],  # NO TOOLS NEEDED - context is in prompt
        system_prompt=final_system_prompt,
        response_format=ToolStrategy(FantasyTeam)
    )

    return agent

# Create a global reference (will be initialized per match)
eco_agent_executor = None

In [22]:
import logging

logging.basicConfig(level=logging.INFO,format="%(asctime)s | %(levelname)s | %(name)s |%(message)s",filename='app.log',filemode='a')
logger=logging.getLogger(__name__)




## Phase 4: Form-Scout Agent (Form-Based Selection)

**Purpose**: Create an AI agent that selects fantasy teams based on player form and venue scoring patterns.

**Agent Strategy**:
1. Fetch venue details & scoring patterns (high-scoring vs low-scoring)
2. Retrieve playing squads with player IDs
3. Fetch recent T20 form for all 22 players (batting SR, bowling wickets)
4. Select 11 players based on momentum and venue alignment

**Tool Execution Order**: venue & squads → batch player stats  
**Selection Criteria**: Player form (SR > 140, wickets) + Venue scoring patterns

In [23]:
from langchain.tools import tool
import requests

# ==========================================
# HELPER FUNCTIONS FOR PLAYER STATS (Form-Scout Only)
# ==========================================

def fetch_single_player_stat(player_id: str) -> dict:
    """Helper function to fetch and parse T20 stats for one player with comprehensive error handling."""
    url = f"https://cricbuzz-cricket.p.rapidapi.com/stats/v1/player/{player_id}"

    try:
        if not HEADERS.get('x-rapidapi-key'):
            logger.error(f"❌ RapidAPI key not configured for player {player_id}")
            return {"id": player_id, "error": "API key not configured"}

        response = requests.get(url, headers=HEADERS, timeout=10)

        # Handle specific HTTP status codes
        if response.status_code == 401:
            logger.error(f"❌ Authentication failed for player {player_id}: 401")
            return {"id": player_id, "error": "API authentication failed"}
        elif response.status_code == 403:
            logger.error(f"❌ Access forbidden for player {player_id}: 403")
            return {"id": player_id, "error": "API access forbidden"}
        elif response.status_code == 404:
            logger.warning(f"⚠️ Player {player_id} not found: 404")
            return {"id": player_id, "error": "Player not found in database"}
        elif response.status_code != 200:
            logger.error(f"❌ API error for player {player_id}: {response.status_code}")
            return {"id": player_id, "error": f"API returned {response.status_code}"}

        response.raise_for_status()  # Raise on 4xx/5xx
        player_data = response.json()

    except requests.Timeout:
        logger.error(f"❌ Timeout fetching player {player_id} (10s)")
        return {"id": player_id, "error": "API request timeout"}
    except requests.ConnectionError as e:
        logger.error(f"❌ Connection error for player {player_id}: {e}")
        return {"id": player_id, "error": f"Connection error: {str(e)}"}
    except requests.RequestException as e:
        logger.error(f"❌ Request error for player {player_id}: {e}")
        return {"id": player_id, "error": f"API request failed: {str(e)}"}
    except ValueError as e:
        logger.error(f"❌ Invalid JSON response for player {player_id}: {e}")
        return {"id": player_id, "error": "Invalid JSON response"}
    except Exception as e:
        logger.error(f"❌ Unexpected error fetching player {player_id}: {e}")
        return {"id": player_id, "error": f"Unexpected error: {str(e)}"}

    # Parse player data with error handling
    t20_batting = []
    t20_bowling = []
    player_name = "Unknown"
    player_role = "Unknown"

    try:
        player_name = player_data.get("name", "Unknown")
        player_role = player_data.get("role", "Unknown")

        # Batting Logic with error handling
        try:
            for row in player_data.get('recentBatting', {}).get('rows', []):
                try:
                    vals = row.get('values', [])
                    if len(vals) > 2 and any(fmt in vals[3] for fmt in ["T20", "T20I"]):
                        score_str = vals[1]
                        if "(" in score_str:
                            raw_runs = score_str.split("(")[0]
                            is_not_out = "*" in score_str
                            clean_runs = raw_runs.replace("*", "").strip()
                            runs = int(clean_runs)
                            balls = int(score_str.split("(")[1].replace(")", ""))
                            sr = round((runs / balls) * 100, 2) if balls > 0 else 0
                            t20_batting.append({
        "runs": runs,
        "balls": balls,
        "sr": sr,
        "status": "Not Out" if is_not_out else "Out",
        "score_display": score_str.split("(")[0].strip() # e.g., "45*" or "45"
    })
                except (ValueError, IndexError) as e:
                    logger.warning(f"⚠️ Error parsing batting stats for {player_name} ({player_id}): {e}")
                    continue
        except Exception as e:
            logger.warning(f"⚠️ Error processing batting data for {player_name} ({player_id}): {e}")

        # Bowling Logic with error handling
        try:
            for row in player_data.get('recentBowling', {}).get('rows', []):
                try:
                    vals = row.get('values', [])
                    if len(vals) > 2 and any(fmt in vals[3] for fmt in ["T20", "T20I"]):
                        bowl_str = vals[1]
                        if "-" in bowl_str:
                            wkts = int(bowl_str.split("-")[0])
                            runs = int(bowl_str.split("-")[1])
                            avg = round(runs / wkts, 2) if wkts > 0 else runs
                            t20_bowling.append({"wkts": wkts, "avg": avg,"match_figure": bowl_str})
                except (ValueError, IndexError) as e:
                    logger.warning(f"⚠️ Error parsing bowling stats for {player_name} ({player_id}): {e}")
                    continue
        except Exception as e:
            logger.warning(f"⚠️ Error processing bowling data for {player_name} ({player_id}): {e}")

    except Exception as e:
        logger.error(f"❌ Unexpected error parsing player {player_id} data: {e}")
        return {"id": player_id, "error": f"Data parsing error: {str(e)}"}

    logger.info(f"✅ Player stats parsed: {player_name} ({player_id})")
    return {
        "id": player_id,
        "name": player_name,
        "role": player_role,
        "bat_form": t20_batting,
        "bowl_form": t20_bowling
    }

@tool
def get_batch_player_stats(player_ids: list) -> str:
    """Fetches T20 form for all players with caching. Caches individual players to avoid redundant API calls."""
    batch_results = []
    errors = []
    successful = 0

    try:
        if not player_ids:
            logger.warning("⚠️ No player IDs provided for batch stats fetch")
            return "Error: Empty player ID list provided"

        logger.info(f"📊 Fetching stats for {len(player_ids)} players...")

        for p_id in player_ids:
            try:
                p_key = f"player_{p_id}"

                # Check cache first (no file I/O per player)
                if cache.exists(p_key):
                    try:
                        cached_value = cache.get(p_key)
                        batch_results.append(cached_value['data'])
                        logger.debug(f"✅ Cache hit for player {p_id}")
                        successful += 1
                        continue
                    except Exception as e:
                        logger.warning(f"⚠️ Cache retrieval error for player {p_id}: {e}")

                # Fetch and cache if not available
                player_stats = fetch_single_player_stat(str(p_id))

                # Check for errors in the returned data
                if "error" in player_stats:
                    errors.append(f"Player {p_id}: {player_stats['error']}")
                    logger.warning(f"⚠️ Error for player {p_id}: {player_stats['error']}")
                else:
                    successful += 1

                try:
                    cache.set(p_key, player_stats)  # Auto-saves with timestamp
                except Exception as e:
                    logger.error(f"❌ Failed to cache player {p_id}: {e}")

                batch_results.append(player_stats)

            except Exception as e:
                logger.error(f"❌ Unexpected error processing player {p_id}: {e}")
                errors.append(f"Player {p_id}: {str(e)}")
                batch_results.append({"id": p_id, "error": str(e)})

        logger.info(f"📊 Batch complete: {successful}/{len(player_ids)} successful")

        if errors:
            logger.warning(f"⚠️ {len(errors)} errors encountered: {', '.join(errors[:3])}")

        return str(batch_results)

    except Exception as e:
        logger.error(f"❌ Critical error in batch player stats: {e}")
        return f"Error: Failed to fetch batch player stats: {str(e)}"

# Define Tool List for Form-Scout Agent (includes cached player stats)
# form_agent_tools = [get_match_venue, get_cricket_squads_by_match, get_batch_player_stats]

In [24]:
import json
import logging
from datetime import datetime

# Assuming IST and cache are defined globally in your environment
# IST = pytz.timezone('Asia/Kolkata')

def get_or_generate_match_context_for_form(match_id: int, match: str) -> dict:
    """
    FORM-SCOUT SNAPSHOT STRATEGY:
    1. Check if a complete form context exists in cache.
    2. If not, invoke the dictionary-based squad tool.
    3. Extract all player IDs directly from the dictionary.
    4. Fetch batch stats for those players.
    5. Save the combined data as a single 'Snapshot' for the Form-Scout agent.
    """
    cache_key = generate_match_context_key(match, "form")

    # --- 1. CACHE CHECK ---
    if cache.exists(cache_key):
        try:
            cached_data = cache.get(cache_key)
            logging.info(f"✅ [FORM SNAPSHOT] Reusing cached context for Match ID: {match_id}")
            return {
                "context": cached_data['data']['context'],
                "is_cached": True,
                "timestamp": cached_data.get('timestamp')
            }
        except Exception as e:
            logging.warning(f"⚠️ Failed to retrieve cached form context: {e}")

    # --- 2. GENERATE FRESH CONTEXT ---
    logging.info(f"📸 [FORM SNAPSHOT] Generating fresh Form context for {match} (ID: {match_id})")

    try:
        # Step 1: Get Squad Data (Invoking the tool that returns a JSON dict)
        # This tool returns: {"match_id": 123, "teams": {"team1": {"players": [{"id": "1", ...}] ...}}}
        squad_json = get_cricket_squads_by_match(match_id)
        squad_data = json.loads(squad_json)

        if "error" in squad_data:
            raise ValueError(f"Squad API Error: {squad_data['error']}")

        # Step 2: Pick Player IDs easily from the structured dictionary
        all_player_ids = []
        for team_key in ["team1", "team2"]:
            team_info = squad_data.get("teams", {}).get(team_key, {})
            for player in team_info.get("players", []):
                if "id" in player:
                    all_player_ids.append(str(player["id"]))

        if not all_player_ids:
            return {
                "context": "Error: No player IDs found. Squad might not be announced.",
                "is_cached": False
            }

        # Step 3: Call Batch Stats with the clean list of IDs
        logging.info(f"📊 Fetching batch stats for {len(all_player_ids)} players...")
        stats_data = get_batch_player_stats.invoke({"player_ids": all_player_ids})

        # Step 4: Build the Final Snapshot String
        # This is the ONLY thing the LLM will read to make its decision.
        full_context = f"""
=== FORM-SCOUT MATCH REASONING SNAPSHOT ===
Generated at: {datetime.now(IST).strftime('%Y-%m-%d %H:%M:%S IST')}
Match: {match} (ID: {match_id})

--- OFFICIAL SQUAD DATA ---
{squad_json}

--- RECENT PLAYER PERFORMANCE (T20 FORM) ---
{stats_data}

=== END SNAPSHOT ===
"""

        # --- 3. STORE AND RETURN ---
        cache.set(cache_key, {"context": full_context})
        logging.info(f"✅ [FORM SNAPSHOT] Successfully cached complete context for {match}")

        return {
            "context": full_context,
            "is_cached": False,
            "generated_at": datetime.now(IST).strftime('%Y-%m-%d %H:%M:%S IST')
        }

    except Exception as e:
        logging.error(f"❌ [FORM SNAPSHOT] Failed to generate context: {e}")
        return {
            "context": f"Critical Error generating Form context: {str(e)}",
            "is_cached": False
        }

In [25]:
from langchain.agents import create_agent

# ==========================================
# FORM-SCOUT AGENT WITH REASONING SNAPSHOT
# ==========================================

form_scout_system_prompt = """Your Role:
You are the "Form-Scout" Agent — a Cricket Fantasy Analyst whose decisions are STRICTLY driven by player form.

CRITICAL: You are operating in SNAPSHOT MODE. A match context snapshot has been pre-generated and cached.
You do NOT need to call tools. Instead, use the provided match context to analyze form and select 11 players.

--- MATCH CONTEXT SNAPSHOT ---
{match_context_snapshot}
--- END SNAPSHOT ---

FORM-BASED SELECTION RULES:
1. Use ONLY the data in the snapshot above (venue, squads, and player stats)
2. Form analysis: Focus on strike rates (SR > 140), wicket-taking records, recent momentum
3. Selection driven by: Player recent form (T20 stats), venue alignment, value picks (all-rounders)
4. Do NOT consider: Past seasons, intuition, or non-provided data
5. Do NOT invent players - use only players from the squads in snapshot

FORM SCORING LOGIC:
- High Strike Rate (SR > 140): Strong batting form indicator
- Wicket Takers (3+ wickets in recent): Top bowlers
- All-rounders: Both batting AND bowling stats in form
- Venue Alignment: Match high-scorers with power venues, bowlers with defensive venues

ROSTER CONSTRAINTS (STRICT):
- Exactly 11 players
- 1 Wicket-Keeper (must have WK role from squads, highest recent T20 runs)
- 4 Bowlers (must have BOWL role from squads, top 4 wicket-takers in recent form)
- 6 Batsmen/All-rounders (BAT or AR roles from squads, prioritize SR > 140)

OUTPUT INSTRUCTIONS:
1. Provide reasoning_summary: Explain player form analysis and venue alignment
2. Select 11 players from the snapshot squads
3. CRITICAL: Include 'player_id' for EVERY player (copy from snapshot)
4. Create FantasyTeam JSON with:
   - reasoning_summary: 3-4 lines explaining form-based selection logic
   - players: List of 11 FantasyPlayer objects (id, name, team, role, scout_logic)

FAILURE RULE:
If any player is missing 'player_id' or does not exist in snapshot squads, selection is INVALID.
"""

def create_form_scout_with_snapshot(match_details: Union[str, dict], agent_type: str = "form"):
    """
    Factory function to create Form-Scout agent or return context for Merging.
    """
    logger.info(f"🔍 Form-Scout initializing for: {match_details['id']} | Mode: {agent_type}")

    # 1. Get or generate match context snapshot
    context_result = get_or_generate_match_context_for_form(match_details['id'], match_details['match'])
    match_context = context_result['context']

    # --- NEW LOGIC FOR OVERALL SCOUT ---
    # If we are just preparing data for the "Overall" merge, stop here and return the string context.
    if agent_type == "overall":
        logger.info(f"📤 Returning match_context only for Overall-Scout merge.")
        return match_context

    # --- ORIGINAL LOGIC FOR FORM SCOUT ---
    is_cached = context_result.get('is_cached', False)
    status = "📦 [CACHED]" if is_cached else "📸 [FRESH]"
    logger.info(f"{status} Form-Scout context ready ({len(match_context)} chars)")

    # Inject snapshot into system prompt
    final_system_prompt = form_scout_system_prompt.format(
        match_context_snapshot=match_context
    )

    # Create agent
    agent = create_agent(
        model=model,
        tools=[],
        system_prompt=final_system_prompt,
        response_format=ToolStrategy(FantasyTeam)
    )

    return agent

# Create a global reference (will be initialized per match)
form_agent_executor = None


## REASONING SNAPSHOT STRATEGY - Usage Guide

### What is Reasoning Snapshot?

Instead of agents calling tools repeatedly for each user, we cache the ENTIRE context once and reuse it.

**First Match Run:**
1. Agents call tools once: venue → squads → weather/stats
2. Entire context cached as "match_context" 
3. Cached: venue info, squads with IDs, weather, pitch analysis

**Subsequent User Requests (Same Match):**
1. Snapshot retrieved from cache (no API calls!)
2. LLM gets context + simple prompt
3. LLM selects 11 players (IDs preserved)
4. Takes: ~1 second per user (vs 10-15 seconds with tool calls)

### Benefits:

✅ **No Rate Limit Issues** - Only 1 API call per match, not per user
✅ **Eliminated Tool Errors** - No 400 "tool choice required" errors
✅ **ID Preservation** - Player IDs visible in prompt context
✅ **Consistency** - All users get same environmental/form context
✅ **Cost Reduced** - 20 users = 1 LLM context call + 20 completion calls (vs 60 tool/LLM calls)

### Code Pattern:

```python
# Create snapshot-based agent for a match
eco_agent = create_eco_scout_with_snapshot("Australia vs Oman")
result = eco_agent.invoke({"messages": [{"role": "user", "content": "Pick the Eco-Scout 11"}]})

# Same match, different user - USES CACHED SNAPSHOT
form_agent = create_form_scout_with_snapshot("Australia vs Oman")
result = form_agent.invoke({"messages": [{"role": "user", "content": "Pick the Form-Scout 11"}]})
```

### Cache Behavior:

- Cache key: `match_context_australia_vs_oman`
- First call: Generates context, caches it
- Subsequent calls on same match: Loads from cache instantly
- New match: Generates fresh context, caches separately


## Phase 5: Match Management & Monitoring

**Purpose**: Fetch today's matches, monitor their status, and trigger agents when matches start.

**Components**:
- `get_todays_match_list()` - Fetch matches from ICC T20 World Cup 2026 series
- `store_daily_matches()` - Smart cache: fetches only if cache is from previous day
- `load_stored_matches()` - Retrieve cached matches
- `check_match_status()` - Poll Cricbuzz API for match state (Preview → In Progress → Result)
- `check_all_matches()` - Group matches into started/upcoming categories

**Scheduler Logic**:
- 7:00 AM: Fetch and store today's matches (one-time)
- Every 5 min: Check which matches have started
- On Match Start: Trigger agent to generate fantasy team

In [26]:
from datetime import datetime

url = "https://cricbuzz-cricket.p.rapidapi.com/matches/v1/upcoming"

def get_todays_match_list():
    response = requests.get(url, headers=HEADERS)
    data = response.json()
    print(data)

    # Get today's date in 'MMM DD' format (e.g., 'Feb 10')
    today_str = datetime.now().strftime("%b %d")

    matches_today = []
    target_series = "Pakistan Super League 2026"

    # Traverse the JSON structure
    for match_type_group in data.get('typeMatches', []):
       for series_group in match_type_group.get('seriesMatches', []):
            wrapper = series_group.get('seriesAdWrapper')

            # FILTER 1: Check if seriesName matches exactly
            if wrapper and wrapper.get('seriesName') == target_series:
                print('inside')
                for match in wrapper.get('matches', []):
                    match_info = match.get('matchInfo', {})
                    status = match_info.get('status', "")
                    print(status)
                    print(today_str)

                    # Check if the match status mentions today's date (Feb 10)
                    if today_str in status:
                        print('inside')
                        team1 = match_info.get('team1', {}).get('teamName')
                        team2 = match_info.get('team2', {}).get('teamName')
                        match_id = match_info.get('matchId')
                        venue = match_info.get('venueInfo', {}).get('ground')
                        city= match_info.get('venueInfo', {}).get('city')

                        matches_today.append({
                            "match": f"{team1} vs {team2}",
                            "id": match_id,
                            "venue": venue,
                            "city":city,
                            "status": status
                        })

    return matches_today

# # Execute and Print
# todays_games = get_todays_match_list()
# todays_games


In [27]:
import json
import requests
from datetime import datetime
from pathlib import Path

# ==============================
# Configuration
# ==============================
MATCH_STORAGE_FILE = "daily_matches.json"


# ==============================
# STEP 1 — Fetch and Store Today's Matches
# ==============================
# ==========================================
# STEP 2 — Load Stored Matches (FROM SHEET)
# ==========================================
def load_stored_matches():
    """
    Checks the Google Sheet. If matches are missing or old,
    calls get_todays_match_list() and updates the sheet.
    """
    try:
        sheet = spreadsheet.worksheet("Today_Matches")
        current_date = datetime.now().strftime("%Y-%m-%d")

        # 1. Read existing data from the sheet
        all_values = sheet.get_all_values()

        # 2. Check if we need to Refresh from API
        # We refresh if: Sheet is empty, ONLY has headers, OR date (Col F) is old
        needs_refresh = False
        if len(all_values) <= 1:
            needs_refresh = True
        else:
            # Check date in Row 2, Column F (index 5)
            last_fetch_date = all_values[1][5] if len(all_values[1]) > 5 else ""
            if last_fetch_date != current_date:
                needs_refresh = True

        # 3. If we need a refresh, call the API and SAVE to sheet
        if needs_refresh:
            print(f"🌐 Cache missing or old. Fetching fresh matches for {current_date}...")
            todays_games = get_todays_match_list() # This is your existing API call

            if not todays_games:
                print("⚠️ No matches found from API today.")
                return []

            # Format for sheet: [ID, Match, Venue, City, Status, FetchDate]
            formatted_rows = []
            for m in todays_games:
                formatted_rows.append([
                    m.get('id'),
                    m.get('match'),
                    m.get('venue'),
                    m.get('city'),
                    m.get('status'),
                    current_date
                ])

            # Clear and update sheet
            sheet.batch_clear(['A2:F20'])
            sheet.insert_rows(formatted_rows, 2)
            print(f"✅ Sheet updated with {len(formatted_rows)} fresh matches.")

            # Re-fetch values to ensure the loop below works on fresh data
            all_values = sheet.get_all_values()

        # 4. Now convert the sheet data into the list of dictionaries for the rest of the script
        cached_matches = []
        for row in all_values[1:]:
            if len(row) >= 5:
                cached_matches.append({
                    "id": row[0],
                    "match": row[1],
                    "venue": row[2],
                    "city": row[3],
                    "status": row[4]
                })

        print(f"📦 Returning {len(cached_matches)} matches for processing.")
        return cached_matches

    except Exception as e:
        print(f"❌ Error in smart match loader: {e}")
        return []

# ==========================================
# STEP 3 — Check Match Status (UNCHANGED)
# ==========================================
def check_match_status(match_id):
    """
    Checks the Google Sheet first. If 'Started' isn't there,
    calls the API and updates the sheet if the status changed.
    """
    sheet = spreadsheet.worksheet("Today_Matches")
    all_values = sheet.get_all_values() # Get all rows

    # --- STEP 1: Check the Sheet Cache ---
    match_row_index = -1
    for i, row in enumerate(all_values):
        if i == 0: continue  # Skip header
        if row[0] == str(match_id):  # Column A is Match_ID
            match_row_index = i + 1  # Sheets are 1-indexed
            current_sheet_status = row[4].lower() # Column E is Status

            # If sheet already knows it started, return immediately (No API call!)
            if current_sheet_status == "started":
                print(f"📦 [CACHE] Match {match_id} already marked as Started in Sheet.")
                return {
                    "match_started": True,
                    "state": row[4],
                    "status": "cached"
                }
            break

    # --- STEP 2: Call the API (Only if Cache says not started) ---
    try:
        print(f"🌐 [API] Checking live status for Match {match_id}...")
        url = f"https://cricbuzz-cricket.p.rapidapi.com/mcenter/v1/{match_id}"
        response = requests.get(url, headers=HEADERS, timeout=10)
        data = response.json()

        state = data.get("state", "").lower()
        status = data.get("status", "").lower()

        # Logic to determine if started
        match_started = (
            state in ["in progress", "live", "innings break"] or
            any(word in status for word in ["opt to", "elect to", "lead by", "need", "won by"])
        )

        # Override for finished/preview
        if state in ["preview", "complete"]:
            match_started = False

        # --- STEP 3: Update the Sheet if status is now 'Started' ---
        if match_started and match_row_index != -1:
            # Update Column E (Status) for this specific match row
            sheet.update_cell(match_row_index, 5, "Started")
            print(f"✅ [SHEET UPDATED] Match {match_id} is now LIVE. Status saved.")

        return {
            "match_started": match_started,
            "state": state,
            "status": status
        }

    except Exception as e:
        print(f"❌ Error checking status: {e}")
        return {"match_started": False, "status": "error", "state": "error"}

# ==========================================
# STEP 4 — Check All Matches (USING SHEET DATA)
# ==========================================
def check_all_matches():
    """Checks status of all matches stored in Google Sheet."""
    # CHANGED: Now calls the sheet-based loader
    matches = load_stored_matches()

    if not matches:
        print("⚠️ No matches found in Today_Matches sheet. Run store_daily_matches first.")
        return [], []

    print(f"\n🔍 Checking {len(matches)} matches from Sheet...\n")

    started_matches = []
    upcoming_matches = []

    for match in matches:
        match_id = match.get("id")
        match_name = match.get("match")

        status_info = check_match_status(match_id)

        # Attach the live status and state to the match object
        match["current_status"] = status_info["status"]
        match["current_state"] = status_info["state"]

        if status_info["match_started"]:
            print(f"🎯 MATCH STARTED → {match_name} ({status_info['status']})")
            started_matches.append(match)
        else:
            print(f"⏳ NOT STARTED  → {match_name} ({status_info['status']})")
            upcoming_matches.append(match)

    return started_matches, upcoming_matches

# ==============================
# CACHE CLEANUP ON STARTUP
# ==============================
print(f"📊 Cache Status: {cache.size()} entries")
cache.cleanup(days=7)

2026-03-28 10:11:37,575 - INFO - ℹ️ Cleanup: No expired entries found (threshold: 7 days)


📊 Cache Status: 0 entries


0

## Phase 6: Agent Orchestration & Supervision

**Purpose**: Create a supervisor agent that manages the workflow and delegates to Eco-Scout or Form-Scout.

**Supervisor Agent (Fantasy Cricket Director)**:
- Verifies only LIVE matches can have teams generated
- Routes requests to appropriate scout (Form-Scout or Eco-Scout)
- Combines both scouts' outputs for "Best Overall Team" selection
- Prevents team generation for upcoming (non-started) matches

**Delegation Tools**:
- `get_started_matches_report()` - Check which matches are currently LIVE
- `call_eco_scout(match)` - Invoke Eco-Scout agent
- `call_form_scout(match)` - Invoke Form-Scout agent

**Workflow**:
1. Check if any matches are live
2. Route user request to appropriate agent(s)
3. Return final fantasy team selection

In [28]:
@tool
def get_started_matches_report() -> dict:
    """
    Checks all daily matches and returns a report of matches that
    have ALREADY STARTED (Live, In Progress, or Complete).
    Automatically fetches today's matches if the cache is empty.
    """
    # 1. Ensure we have the list of matches for today
    # This checks cache first, then API if necessary
    matches = load_stored_matches()

    # Case A: No matches were found in the schedule for today at all
    if not matches:
        return "📅 No matches are scheduled for today in the tournament calendar."

    # 2. Check the real-time status of the scheduled matches
    # This triggers the API check for 'state' and 'status'
    started, upcoming = check_all_matches()

    # Case B: Matches exist in the schedule, but none have kicked off yet
    if not started:
        return (
            f"⏳ No matches have started yet. There are {len(upcoming)} matches "
            "scheduled for later today, currently in 'Upcoming' status."
        )

    # Case C: Success - Format the report for matches that are active or done
    return started
@tool
def call_form_scout(match_details: Union[str, dict]) -> str:
    """
    Delegates to the Form-Scout Expert using snapshot-based context.
    Args:
        match_details: A dictionary or JSON string containing 'match', 'id', 'venue', and 'city'.
    """

    agent_input = {
        "messages": [
            {
                "role": "user",
                "content": f"Pick the Form-Scout 11 for this match: {match_details}."
            }
        ]
    }

    try:
        form_agent = create_form_scout_with_snapshot(match_details,'form')
        response = form_agent.invoke(agent_input)
        print(response)

        if "messages" in response and len(response["messages"]) > 0:
            final_message = response["messages"][-1]
            if hasattr(final_message, 'content') and final_message.content:
                return final_message.content

        return "Form-Scout was unable to generate a team. Check if player data is available."

    except Exception as e:
        return f"Error in Form-Scout delegation: {str(e)}"

@tool
def call_eco_scout(match_details: Union[str, dict]) -> str:
    """
    Delegates to the Form-Scout Expert using snapshot-based context.
    Args:
        match_details: A dictionary or JSON string containing 'match', 'id', 'venue', and 'city'.
    """
    agent_input = {
        "messages": [
            {
                "role": "user",
                "content": f"Pick the Eco-Scout 11 for {match_details}."
            }
        ]
    }

    try:
        eco_agent = create_eco_scout_with_snapshot(match_details,'eco')
        response = eco_agent.invoke(agent_input)

        if "messages" in response and len(response["messages"]) > 0:
            final_message = response["messages"][-1]
            return final_message.content

        return "Eco-Scout failed to provide a readable response."

    except Exception as e:
        return f"Error delegating to Eco-Scout: {str(e)}"

In [29]:
MERGE_PROMPT = """You are the "Ultimate Fantasy Director".
Your task is to build a winning 'Playing 11' by merging two specialized data streams:
1. PLAYER FORM DATA (Recent performances, strike rates, wickets).
2. ECO/VENUE DATA (Pitch behavior, weather, venue dimensions).

CRITICAL RULES:
1. DATA SOURCE: Only use players provided in the input text. Never hallucinate players.
2. BALANCE: Select 11 players. Ensure a valid team balance (Wicketkeepers, Batsmen, All-rounders, Bowlers).
3. LOGIC: Compare the 'Form' (who is playing well) with 'Eco' (whose style fits this specific pitch).
   - Example: A fast bowler with great 'Form' might be dropped if the 'Eco' data says the pitch is a spinning track.
4. IDENTIFICATION: You MUST preserve the exact 'player_id' for every player selected.
5. FORMAT: Return a single JSON object.

ROSTER CONSTRAINTS (STRICT):
- Exactly 11 players
- 1 Wicket-Keeper (must have WK role from squads, highest recent T20 runs)
- 4 Bowlers (must have BOWL role from squads, top 4 wicket-takers in recent form)
- 6 Batsmen/All-rounders (BAT or AR roles from squads, prioritize SR > 140)

OUTPUT INSTRUCTIONS:
1. Provide reasoning_summary: Explain player form analysis and venue alignment
2. Select 11 players from the snapshot squads
3. CRITICAL: Include 'player_id' for EVERY player (copy from snapshot)
4. Create FantasyTeam JSON with:
   - reasoning_summary: 3-4 lines explaining form-based selection logic
   - players: List of 11 FantasyPlayer objects (id, name, team, role, scout_logic)

FAILURE RULE:
If any player is missing 'player_id' or does not exist in snapshot squads, selection is INVALID.
"""

# Create the Merge Agent
merge_agent = create_agent(
    model=model,
    system_prompt=MERGE_PROMPT,
    response_format=ToolStrategy(FantasyTeam)
)
def fantasy_team_router(user_query: str):
    """
    Highly optimized router.
    Form/Eco cases = 1 LLM Call.
    Overall case = 1 LLM Call (By fetching raw data instead of calling scout agents).
    """
    query_lower = user_query.lower()

    # --- CASE 0: Match List (0 LLM Calls) ---
    if any(word in query_lower for word in ["matches", "today", "show", "available"]):
        return get_started_matches_report.invoke({})

    # --- PRE-CHECK: Get Live Matches ---
    live_matches = get_started_matches_report.invoke({})
    if isinstance(live_matches, str):
            return live_matches  # Returns the "⏳ No matches have started yet..." message to the user

        # 2. Check if the list is empty (just in case)
    if not live_matches or len(live_matches) == 0:
        return "⚠️ I couldn't find any active matches in the system right now."
    # Auto-pick the first match
    target_match = live_matches[0]

    # --- CASE 1: Form-Scout (1 LLM Call) ---
    if "form" in query_lower:
        print(f"🚀 Case 1: Form-Scout LLM Call...")
        # This calls the agent which thinks and returns a team
        return call_form_scout.invoke({"match_details": target_match})

    # --- CASE 2: Eco-Scout (1 LLM Call) ---
    if "eco" in query_lower:
        print(f"🚀 Case 2: Eco-Scout LLM Call...")
        # This calls the agent which thinks and returns a team
        return call_eco_scout.invoke({"match_details": target_match})

    # --- CASE 3: Best Overall Team (ONLY 1 LLM Call) ---
    if any(word in query_lower for word in ["best", "overall", "merge"]):
        print(f"🚀 Case 3: Overall-Scout (1 Merge Call Only)...")

        # FETCH RAW DATA: We bypass the LLM thinking here.
        # We just get the 'match_context' strings from your factory functions.
        form_context = create_form_scout_with_snapshot(target_match, agent_type="overall")
        eco_context = create_eco_scout_with_snapshot(target_match, agent_type="overall")

        # COMBINE & CALL MERGE AGENT: This is the ONLY LLM call for Case 3.
        merge_input = f"""
        SNAPSHOT 1 (Player Form & Stats):
        {form_context}

        SNAPSHOT 2 (Pitch, Weather & Venue):
        {eco_context}
        """

        final_result = merge_agent.invoke({"messages": [("user", merge_input)]})

        # Extract structured JSON
        if "structured_response" in final_result:
             return final_result["structured_response"].model_dump_json(indent=2)

        return final_result

    return "Please specify: Form-Scout, Eco-Scout, or Overall-Scout team."

In [30]:
user_input = "pick eco 11"
response = fantasy_team_router(user_input)

print(f"🤖 Response:\n{response}")
if any(word in user_input for word in ["best", "overall", "merge", "both"]):
    agent_type = "Overall-Scout"
elif "eco" in user_input:
    agent_type = "Eco-Scout"
else:
    # Defaulting to Form-Scout if no other keywords are found
    agent_type = "Form-Scout"
# log_agent_selection('player_7',agent_type,response)

📦 Returning 1 matches for processing.
📦 Returning 1 matches for processing.

🔍 Checking 1 matches from Sheet...



2026-03-28 10:11:40,089 - INFO - 🔍 Eco-Scout initializing for: Rawalpindi Pindiz vs Peshawar Zalmi | Mode: eco
2026-03-28 10:11:40,090 - INFO - 📸 [SNAPSHOT] Generating fresh match context for Rawalpindi Pindiz vs Peshawar Zalmi


📦 [CACHE] Match 148973 already marked as Started in Sheet.
🎯 MATCH STARTED → Rawalpindi Pindiz vs Peshawar Zalmi (cached)
🚀 Case 2: Eco-Scout LLM Call...


2026-03-28 10:11:43,271 - INFO - ✅ Weather for Lahore retrieved and cached
2026-03-28 10:11:44,796 - INFO - ✅ Pitch report retrieved and cached for venue_name Gaddafi Stadium + city Lahore
2026-03-28 10:11:44,813 - INFO - ✅ [SNAPSHOT] Cached complete match context (5222 chars)
2026-03-28 10:11:44,814 - INFO - 📸 [FRESH] Eco-Scout context ready (5222 chars)
2026-03-28 10:11:46,939 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


🤖 Response:
Returning structured response: reasoning_summary='The Gaddafi Stadium pitch is known for being batting-friendly with high scores. The weather is partially cloudy, which might slightly favor bowlers but generally supports batsmen. I have selected a team with a balance of strong batsmen, all-rounders, and bowlers.' players=[FantasyPlayer(player_id='8359', name='Babar Azam', team='Peshawar Zalmi', role='Batsman', scout_logic='Strong batsman for a batting-friendly pitch'), FantasyPlayer(player_id='15038', name='Mohammad Haris', team='Peshawar Zalmi', role='WK-Batsman', scout_logic='Wicket-keeper batsman for a high-scoring match'), FantasyPlayer(player_id='9494', name='Kusal Mendis', team='Peshawar Zalmi', role='WK-Batsman', scout_logic='Additional wicket-keeper batsman option'), FantasyPlayer(player_id='13170', name='Aaron Hardie', team='Peshawar Zalmi', role='Batting Allrounder', scout_logic='All-rounder for balance and batting depth'), FantasyPlayer(player_id='9551', name='Mi

In [ ]:
import time

def update_agent_responses():
    # ── 1. Load Player_agentsheet ──────────────────────────────────────────
    player_sheet = spreadsheet.worksheet("Player_agent")
    all_values = player_sheet.get_all_values()
    headers = all_values[0]
    data_rows = all_values[1:]

    print("Headers found:", headers)
    print(f"Total players found: {len(data_rows)}")

    # ── 2. Prepare response sheet ──────────────────────────────────────────
    response_sheet = spreadsheet.worksheet("Agent_Response")
    response_sheet.batch_clear(["A2:D1000"])

    rows_to_write = []

    # ── 3. Loop through each player ────────────────────────────────────────
    for i, row in enumerate(data_rows):
        if not any(row):  # skip empty rows
            continue

        player_id  = row[0]
        name       = row[1]
        match      = row[2]
        match_id   = row[3]
        agent_type = row[4].strip().upper()

        print(f"\n[{i+1}/{len(data_rows)}] Processing {player_id} - {name}...")

        # Build user_input
        if agent_type == "OVERALL":
            user_input = "pick overall 11"
        elif agent_type == "ECO":
            user_input = "pick eco 11"
        else:
            user_input = "pick form 11"

        # ── 4. Call the router ─────────────────────────────────────────────
        try:
            response = fantasy_team_router(user_input)
        except Exception as e:
            response = f"ERROR: {e}"
        if "No matches have started yet" in str(response) or "scheduled for later" in str(response):
                print(f"🛑 STOPPING: {response}")
                # Optional: You can still write this one error row so you know why it stopped
                rows_to_write.append([player_id, "System", response, "N/A"])
                break
        # ── 5. Determine final agent_type label ────────────────────────────
        if any(word in user_input for word in ["best", "overall", "merge", "both"]):
            final_agent_type = "Overall-Scout"
        elif "eco" in user_input:
            final_agent_type = "Eco-Scout"
        else:
            final_agent_type = "Form-Scout"

        print(f"✅ Player {player_id} | {final_agent_type} | Match {match_id}")

        rows_to_write.append([player_id, final_agent_type, response, match_id])

        # ── 6. Sleep between players (skip on last player) ─────────────────
        if i < len(data_rows) - 1:
            print("⏳ Waiting 2 seconds before next player...")
            time.sleep(2)

    # ── 7. Write all rows at once ──────────────────────────────────────────
    if rows_to_write:
        response_sheet.insert_rows(rows_to_write, 2)
        print(f"\n🎉 Done! Stored responses for {len(rows_to_write)} players.")
    else:
        print("⚠️ No players found in Player_agent sheet.")


if __name__ == "__main__":
    update_agent_responses()

Headers found: ['player_id', 'Name', 'Match', 'Match_id', 'agent_type']
Total players found: 50

[1/50] Processing PLR_1001 - Arun Kumar...


📦 Returning 1 matches for processing.
📦 Returning 1 matches for processing.

🔍 Checking 1 matches from Sheet...



2026-03-28 09:41:03,322 - INFO - 🔍 Eco-Scout initializing for: Peshawar Zalmi vs Gaddafi Stadium | Mode: eco
2026-03-28 09:41:03,324 - INFO - 📸 [SNAPSHOT] Generating fresh match context for Peshawar Zalmi vs Gaddafi Stadium


📦 [CACHE] Match 148973 already marked as Started in Sheet.
🎯 MATCH STARTED → Peshawar Zalmi vs Gaddafi Stadium (cached)
🚀 Case 2: Eco-Scout LLM Call...


2026-03-28 09:41:06,021 - INFO - ✅ Weather for Lahore retrieved and cached
2026-03-28 09:41:07,882 - INFO - ✅ Pitch report retrieved and cached for venue_name Gaddafi Stadium + city Lahore
2026-03-28 09:41:07,903 - INFO - ✅ [SNAPSHOT] Cached complete match context (5220 chars)
2026-03-28 09:41:07,905 - INFO - 📸 [FRESH] Eco-Scout context ready (5220 chars)
2026-03-28 09:41:10,471 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


🤖 Response:
Returning structured response: reasoning_summary='The pitch at Gaddafi Stadium is batting-friendly with an average 1st innings score of 253 runs. The weather is partially cloudy, which might slightly favor bowlers but overall conditions are high-scoring. I have selected a team with a balance of batsmen, all-rounders, and bowlers to adapt to these conditions.' players=[FantasyPlayer(player_id='8359', name='Babar Azam', team='Peshawar Zalmi', role='Batsman', scout_logic='Batting-friendly pitch, a key batsman'), FantasyPlayer(player_id='15038', name='Mohammad Haris', team='Peshawar Zalmi', role='WK-Batsman', scout_logic='Wicket-keeper batsman for flexibility'), FantasyPlayer(player_id='9494', name='Kusal Mendis', team='Peshawar Zalmi', role='WK-Batsman', scout_logic='Additional wicket-keeper batsman option'), FantasyPlayer(player_id='13170', name='Aaron Hardie', team='Peshawar Zalmi', role='Batting Allrounder', scout_logic='All-rounder for batting depth'), FantasyPlayer(player